<a href="https://colab.research.google.com/github/avani-joshii/ml-object-classifier/blob/main/ML_Object_Classifier_1_smaller_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print("All imports done!", torch.__version__)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

with zipfile.ZipFile('Dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

for folder in os.listdir('/content/dataset'):
    path = f'/content/dataset/{folder}'
    if os.path.isdir(path):
        count = len(os.listdir(path))
        print(f"{folder}: {count} images")

In [ ]:
import zipfile
import os

with zipfile.ZipFile('Dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

# Check structure
for folder in os.listdir('/content/Dataset'):
    path = f'/content/Dataset/{folder}'
    if os.path.isdir(path):
        count = len(os.listdir(path))
        print(f"{folder}: {count} images")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Resize, convert to tensor, normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Load dataset
full_dataset = datasets.ImageFolder('/content/Dataset', transform=transform)

# Split into 80% train, 20% test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=False)

print(f"Total images: {len(full_dataset)}")
print(f"Training: {len(train_data)}")
print(f"Testing: {len(test_data)}")
print(f"Classes: {full_dataset.classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))
class_names = full_dataset.classes

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = images[i].numpy().transpose(1, 2, 0)
    img = (img * 0.5) + 0.5  # unnormalize
    ax.imshow(img)
    ax.set_title(class_names[labels[i]])
    ax.axis('off')

plt.suptitle("Sample images from your dataset")
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 3 color channels → 16 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # shrink image by half

            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 16 → 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # shrink again
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, 4)                            # 4 classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, loss_fn, optimizer, epochs=10):
    train_losses = []
    train_accuracies = []

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            outputs = model(images)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predicted = outputs.argmax(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss / len(train_loader)
        accuracy = 100 * correct / total
        train_losses.append(avg_loss)
        train_accuracies.append(accuracy)
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.2f}%")

    return train_losses, train_accuracies

train_losses, train_accuracies = train(model, train_loader, loss_fn, optimizer, epochs=10)

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        predicted = outputs.argmax(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
class CNNWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Dropout(0.5),  # randomly switches off 50% of neurons
            nn.Linear(128, 4)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_dropout = CNNWithDropout()
optimizer_dropout = optim.Adam(model_dropout.parameters(), lr=0.001)

losses_dropout, accs_dropout = train(model_dropout, train_loader, loss_fn, optimizer_dropout, epochs=10)

In [ ]:
model_dropout.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model_dropout(images)
        predicted = outputs.argmax(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy with Dropout: {100 * correct / total:.2f}%")


In [ ]:
# Test accuracy for original model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        predicted = outputs.argmax(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
test_acc_original = 100 * correct / total

# Test accuracy for dropout model
model_dropout.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model_dropout(images)
        predicted = outputs.argmax(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
test_acc_dropout = 100 * correct / total

# Plot train loss comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, marker='o', label='Without Dropout')
plt.plot(losses_dropout, marker='o', label='With Dropout')
plt.title("Train Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.bar(['Without Dropout', 'With Dropout'],
        [test_acc_original, test_acc_dropout],
        color=['blue', 'orange'])
plt.title("Test Accuracy Comparison")
plt.ylabel("Accuracy %")
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Without Dropout - Test Accuracy: {test_acc_original:.2f}%")
print(f"With Dropout - Test Accuracy: {test_acc_dropout:.2f}%")

In [ ]:
from torch.utils.data import DataLoader, random_split

transform_augmented = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Reload dataset with augmentation
augmented_dataset = datasets.ImageFolder('/content/Dataset', transform=transform_augmented)
train_size = int(0.8 * len(augmented_dataset))
test_size = len(augmented_dataset) - train_size
train_aug, test_aug = random_split(augmented_dataset, [train_size, test_size])

train_loader_aug = DataLoader(train_aug, batch_size=8, shuffle=True)
test_loader_aug = DataLoader(test_aug, batch_size=8, shuffle=False)

# Train fresh model with augmented data
model_aug = SimpleCNN()
optimizer_aug = optim.Adam(model_aug.parameters(), lr=0.001)

losses_aug, accs_aug = train(model_aug, train_loader_aug, loss_fn, optimizer_aug, epochs=10)

In [ ]:
model_aug.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader_aug:
        outputs = model_aug(images)
        predicted = outputs.argmax(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy with Augmentation: {100 * correct / total:.2f}%")